# 00 - Limpeza e Setup

Remove conteúdo da pasta **data** para subir novamente os conteúdos definidos abaixo na etapa **Geração de Dados Transacionais**

In [1]:
import os
import shutil

# 1. Descobre a raiz do projeto independentemente de onde o Jupyter foi aberto
# Pega o diretório atual de execução
diretorio_atual = os.getcwd()

# Se estiver dentro da pasta 'notebook', volta uma para a raiz. Se já estiver na raiz, mantém.
if diretorio_atual.endswith('notebook'):
    raiz_projeto = os.path.dirname(diretorio_atual)
else:
    raiz_projeto = diretorio_atual

# Cria o caminho absoluto exato para a pasta data
PASTA_DADOS = os.path.join(raiz_projeto, 'data')

print(f"Caminho exato da pasta de dados: {PASTA_DADOS}")

# 2. Deleta a pasta se ela já existir
if os.path.exists(PASTA_DADOS):
    print(f"Limpando pasta {PASTA_DADOS} antiga...")
    shutil.rmtree(PASTA_DADOS)

# 3. Cria a pasta novamente
os.makedirs(PASTA_DADOS, exist_ok=True)
print("Pasta preparada para novos dados.\n")

Caminho exato da pasta de dados: /home/ian/spark-delta-minio-sqlserver/data
Limpando pasta /home/ian/spark-delta-minio-sqlserver/data antiga...
Pasta preparada para novos dados.



## 1. Geração de Dados Transacionais (E-commerce)

Nesta etapa, utilizamos a biblioteca **Pandas** para criar conjuntos de dados simulados (*Mock Data*). Esses dados representam o sistema de origem (banco de dados transacional) de uma loja online.

O modelo de dados foi estruturado respeitando relacionamentos essenciais:
* **Catálogo:** Tabelas de `categorias` e `produtos`.
* **Usuários:** Tabela de `clientes` com informações de localização e status.
* **Transações (Fatos):** Tabelas de `vendas` (cabeçalho da compra) e `itens_venda` (detalhamento dos produtos adquiridos).

Após a criação dos DataFrames em memória, o script itera sobre eles e os exporta como arquivos `.csv` para a nossa pasta local `data/`. O parâmetro `index=False` é utilizado para não exportar a coluna de controle numérico do Pandas, mantendo os arquivos limpos e prontos para a ingestão no SQL Server.

In [2]:
import pandas as pd
import os

# 1. Definição dos DataFrames com Pandas
dataframes = {
    'categorias.csv': pd.DataFrame([
        [1, 'Eletrônicos', 'Smartphones, notebooks e gadgets'],
        [2, 'Vestuário', 'Roupas masculinas e femininas'],
        [3, 'Livros', 'Ficção, técnicos e literatura'],
        [4, 'Casa', 'Móveis e itens de decoração'],
        [5, 'Esportes', 'Artigos esportivos e suplementos']
    ], columns=['id_categoria', 'nome_categoria', 'descricao']),

    'produtos.csv': pd.DataFrame([
        [1, 'Smartphone XYZ', 1, 1500.00, 50],
        [2, 'Notebook Pro', 1, 3500.00, 30],
        [3, 'Camiseta Básica', 2, 50.00, 100],
        [4, 'Calça Jeans', 2, 120.00, 80],
        [5, 'O Senhor dos Anéis', 3, 60.00, 40],
        [6, 'Livro Spark e Delta', 3, 85.00, 60],
        [7, 'Sofá Retrátil', 4, 1200.00, 10],
        [8, 'Mesa de Jantar', 4, 800.00, 15],
        [9, 'Bola de Futebol', 5, 80.00, 50],
        [10, 'Tênis de Corrida', 5, 250.00, 40]
    ], columns=['id_produto', 'nome', 'id_categoria', 'preco', 'estoque']),

    'clientes.csv': pd.DataFrame([
        [1, 'João Silva', 'SP', 'Ativo'],
        [2, 'Maria Oliveira', 'RJ', 'Ativo'],
        [3, 'Carlos Santos', 'MG', 'Inativo'],
        [4, 'Ana Costa', 'BA', 'Ativo'],
        [5, 'Pedro Alves', 'PR', 'Ativo'],
        [6, 'Fernanda Lima', 'RS', 'Inativo'],
        [7, 'Lucas Gomes', 'PE', 'Ativo'],
        [8, 'Juliana Rocha', 'SC', 'Ativo'],
        [9, 'Marcos Dias', 'CE', 'Ativo'],
        [10, 'Camila Mendes', 'GO', 'Ativo']
    ], columns=['id_cliente', 'nome', 'estado', 'status_conta']),

    'vendas.csv': pd.DataFrame([
        [1, 1, '2026-05-01', 1500.00],
        [2, 2, '2026-05-02', 3500.00],
        [3, 4, '2026-05-02', 170.00],
        [4, 5, '2026-05-03', 145.00],
        [5, 7, '2026-05-03', 1200.00],
        [6, 8, '2026-05-04', 800.00],
        [7, 9, '2026-05-04', 330.00],
        [8, 10, '2026-05-05', 250.00],
        [9, 1, '2026-05-05', 120.00],
        [10, 2, '2026-05-06', 60.00]
    ], columns=['id_venda', 'id_cliente', 'data_venda', 'valor_total']),

    'itens_venda.csv': pd.DataFrame([
        [1, 1, 1, 1, 1500.00],
        [2, 2, 2, 1, 3500.00],
        [3, 3, 3, 1, 50.00],
        [4, 3, 4, 1, 120.00],
        [5, 4, 5, 1, 60.00],
        [6, 4, 6, 1, 85.00],
        [7, 5, 7, 1, 1200.00],
        [8, 6, 8, 1, 800.00],
        [9, 7, 9, 1, 80.00],
        [10, 7, 10, 1, 250.00],
        [11, 8, 10, 1, 250.00],
        [12, 9, 4, 1, 120.00],
        [13, 10, 5, 1, 60.00]
    ], columns=['id_item', 'id_venda', 'id_produto', 'quantidade', 'preco_unitario'])
}

# 2. Exportação para CSV usando a variável PASTA_DADOS da primeira célula
print("A gerar os ficheiros de E-commerce com Pandas...")

for nome_ficheiro, df in dataframes.items():
    caminho = os.path.join(PASTA_DADOS, nome_ficheiro)
    df.to_csv(caminho, index=False, encoding='utf-8-sig')
    print(f" {nome_ficheiro} exportado! ({len(df)} registos)")

print("\nProcesso concluído com sucesso!")

A gerar os ficheiros de E-commerce com Pandas...
 categorias.csv exportado! (5 registos)
 produtos.csv exportado! (10 registos)
 clientes.csv exportado! (10 registos)
 vendas.csv exportado! (10 registos)
 itens_venda.csv exportado! (13 registos)

Processo concluído com sucesso!
